In [ ]:
from pathlib import Path
from utils.data_preparation import (
    find_images,
    train_test_split_paths,
    partial_fit_chars,
    evaluate
)

# 1. Daten einlesen
data_char_dir:Path=Path("../../train_data/data_chars/")
if data_char_dir.exists():
    image_paths = find_images(root=data_char_dir)


In [ ]:

train_paths, test_paths = train_test_split_paths(image_paths)

In [ ]:


# 2. Neues Modell trainieren
model_out = Path("models/sgd_chars_v1.joblib")
pipe, le = partial_fit_chars(
    train_paths,
    batch_size=1024,
    model_out=model_out
)


In [ ]:

# 3. Evaluieren
results = evaluate(pipe, le, test_paths)
print(results["report"])


In [ ]:

from utils.model_prediction import predict_digit_from_path


out = predict_digit_from_path(
    Path("models/sgd_chars_v1.joblib"),
    Path("../../train_data/data_chars/1/arial_1_108.png")
)
print(out)

In [2]:
import requests
import base64
import zipfile
import io
import pandas as pd

#1: Preparing the URL.
base_url = "https://www.kaggle.com/api/v1"
owner_slug = "thomassedlmeyr"
dataset_slug = "/erman-character-recognition-dataset"
dataset_version = "1"

url = f"{base_url}/datasets/download/{owner_slug}/{dataset_slug}?datasetVersionNumber={dataset_version}"

#2: Encoding the credentials and preparing the request header.
username = "arbolsito"
key = "04f5e52128a4a2fee92163d9fda87ad1"
creds = base64.b64encode(bytes(f"{username}:{key}", "ISO-8859-1")).decode("ascii")
headers = {
  "Authorization": f"Basic {creds}"
}

#3: Sending a GET request to the URL with the encoded credentials.
response = requests.get(url, headers=headers)

#4: Loading the response as a file via io and opening it via zipfile.
zf = zipfile.ZipFile(io.BytesIO(response.content))


BadZipFile: File is not a zip file

In [ ]:

#5: Reading the CSV from the zip file and converting it to a dataframe.
test_data = "test.csv"
train_data = "train.csv"
df_train = pd.read_csv(zf.open(train_data))
df_test = pd.read_csv(zf.open(test_data))


In [ ]:

#6: Printing the dataframe. Finally!
print(df_train[1:20])
print(df_test.head())

In [ ]:

# 4. Weitertrainieren mit neuen Daten
new_data_dir = Path("data_digits/new_samples")
new_paths = find_images(new_data_dir)

pipe2, le2 = partial_fit_chars(
    new_paths,
    existing_model=model_out,
    model_out=Path("models/sgd_chars_v2.joblib")
)